# Active Learning in Machine Learning - Starter Notebook
This notebook introduces an active learning loop where a model iteratively queries uncertain points for labeling.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    class_sep=1.0,
    random_state=42,
)

X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rng = np.random.default_rng(42)
initial_idx = rng.choice(len(X_pool), size=30, replace=False)

labeled_mask = np.zeros(len(X_pool), dtype=bool)
labeled_mask[initial_idx] = True

In [ ]:
history = []
for step in range(10):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_pool[labeled_mask], y_pool[labeled_mask])
    test_acc = accuracy_score(y_test, model.predict(X_test))
    history.append({'step': step, 'labeled_count': labeled_mask.sum(), 'test_accuracy': test_acc})

    unlabeled_idx = np.where(~labeled_mask)[0]
    probs = model.predict_proba(X_pool[unlabeled_idx])[:, 1]
    uncertainty = np.abs(probs - 0.5)
    query_idx = unlabeled_idx[np.argsort(uncertainty)[:20]]
    labeled_mask[query_idx] = True

pd.DataFrame(history).round(3)